# Lesson 01 - 人工智能代理簡介

歡迎來到<strong>初學者人工智能代理</strong>課程的第一課！

<strong>人工智能代理</strong>是一個使用大型語言模型（LLM）作為推理引擎的程式，能夠在真實世界中採取<em>行動</em> — 呼叫 API、查詢資料庫或執行程式碼 — 以用戶名義達成目標。

在這個筆記本中，你將建立你的第一個代理：一個推薦假期目的地的<strong>旅遊代理</strong>。在過程中你將學習如何：

1. 使用 **Microsoft Agent Framework** 連接到 Microsoft Foundry 代理服務。
2. 給代理一個 <strong>工具</strong> — 一個它可以呼叫的純 Python 函數。
3. 執行代理並檢查其回應。
4. 逐字元串流代理的回應。


## 設定

在執行此筆記本之前，請確保您已：

1. **擁有一個 Microsoft Foundry 項目** 並已部署聊天模型（例如 `gpt-4.1-mini`）。
2. **已使用 Azure CLI 登入** — 在終端機中執行 `az login`。
3. **設定所需的環境變數：**
   - `AZURE_AI_PROJECT_ENDPOINT` — 您的 Microsoft Foundry 項目端點。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 您部署的模型名稱。

下面的程式碼格會安裝您所需的 Python 套件。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## 建立你的第一個代理人

一個代理人需要兩樣東西：

- <strong>指示</strong>，告訴它<em>它是誰</em>以及<em>如何行事</em>（系統提示）。
- <strong>工具</strong> — 使用 `@tool` 裝飾的 Python 函數，代理人可以呼叫這些函數來取得資訊或執行動作。

以下我們定義了一個簡單的工具，回傳一個熱門旅遊地點清單。當使用者要求旅遊建議時，代理人將使用這個工具。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## 串流回應

為了獲得更具互動性的體驗，你可以<strong>串流</strong>代理的回應。代理不需等待完整回覆，即可隨著文字片段生成逐步輸出。這在聊天介面中特別有用，因為你希望即時顯示輸出結果。


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## 總結

在本課程中，你學會了如何：

- <strong>建立一個提供者</strong>，通過 `FoundryChatClient` 連接到 Microsoft Foundry Agent Service。
- **使用 `@tool` 裝飾器定義一個工具**，讓代理可以呼叫你的 Python 函數。
- <strong>以使用者訊息執行代理</strong> 並列印其回應。
- <strong>串流回應</strong> 以實現即時輸出。

在下一課中，我們將更深入探討代理框架，並學習如何賦予代理更強大的工具和多步推理能力。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
